# Supplementary Figure 6 — d2 as a robust indicator of the NREM CA1 regime

The barrage share of a window's NREM events, barrages / (barrages + SWRs), predicted from the background state between events (every 40 ms bin overlapping a ripple or a barrage removed; CA1 units; non-overlapping 30 s NREM windows; 11 barrage sessions). d2 = d2 (2024 Euclidean, AR p = 20, 40 ms).

- **a** One binomial model per session, the five background measures z-scored within session and entered together; coefficient = change in log-odds of the barrage share per within-session SD; one-sample signed-rank of the session coefficients against 0 (`r16_block_level_dissociation.py`).
- **b** Stability across 45 blocks of 30 min: random-effects pooling of the per-block coefficients; |μ|/τ = pooled effect ÷ between-block SD. Brackets: d2 vs each measure, paired session bootstrap (2 000 resamples), two-sided, Holm.
- **c** Transfer to a held-out session (leave-one-session-out): held-out ÷ in-sample deviance explained.
- b, c: boxes = 2 000 session-level cluster-bootstrap resamples, whiskers 95 % interval, diamond = full-data estimate (`r16_robustness_index.py`).

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('.'))   # notebooks/figures_revision
import final_figs as F                     # paths, save/copy helpers; adds the revision folder to sys.path

## a–c

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import fig2_new as N
import r16_robustness_index as RI
from r16_predictor_heterogeneity import ORTHO_CSV, LAB_ORTHO

# Supplementary Fig 6 -- d2 as a robust indicator of the NREM CA1 regime: the barrage share of a window's NREM events
# (barrages / (barrages + SWRs)) from the background state between events (every 40 ms bin overlapping a ripple or a
# barrage removed; CA1 units; non-overlapping 30 s NREM windows; barrage cohort, 11 sessions -- NN8_day6 has 6
# barrages). Panels a and b, c are redrawn from their stored tables. Colours as Fig. 2's
# GLM (Fig. 2e: d2 red, MUA grey, the variability measure pink, E/I brown) plus a light grey for synchrony.
COL = {"d2": N.RED, "MUA": N.GREY, "fano": N.PINK, "EI": N.DARK, "sync": "#c7c7c7"}
LOG = RI.LOGDIR
key = lambda c: c[:-3] if c.endswith("_bg") else c       # noqa: E731  d2_bg -> d2

# ---- a: one binomial model per session (barrage share of the window's events ~ the five background measures,
# z-scored within session, entered together); coefficient per within-session SD; one-sample signed-rank over sessions
# (r16_block_level_dissociation.py: r16_event_type_classifier_session{,_summary}.csv)
C = pd.read_csv(os.path.join(LOG, "r16_event_type_classifier_session.csv"))
S = pd.read_csv(os.path.join(LOG, "r16_event_type_classifier_session_summary.csv")).set_index("predictor")
ORDER = ["d2_bg", "MUA_bg", "fano_bg", "sync_bg", "EI_bg"]
LAB = {"d2_bg": "d2", "MUA_bg": "MUA", "fano_bg": "MUA Fano", "sync_bg": "Synchrony", "EI_bg": "E/I"}
for c in ORDER:
    v = C[C.predictor == c].coef
    assert abs(stats.wilcoxon(v).pvalue - S.loc[c, "p_signrank"]) < 1e-9, c
_v = C[C.predictor == "d2_bg"].coef
print(f"a: d2 same sign in {int((np.sign(_v) == np.sign(_v.median())).sum())} of {len(_v)} sessions; one-sample two-sided "
      "Wilcoxon signed-rank: " + "; ".join(f"{LAB[c]} p = {S.loc[c, 'p_signrank']:.2g}" for c in ORDER))
F.final_style()
fig, ax = plt.subplots(figsize=(56 * F.MM, 50 * F.MM), layout="constrained")
YL = (-2.0, 2.0)
rng = np.random.default_rng(1)
for i, c in enumerate(ORDER):
    v = C[C.predictor == c].coef.to_numpy()
    bp = ax.boxplot(v, positions=[i], widths=0.55, showfliers=False, patch_artist=True,
                    medianprops=dict(color="k", lw=0.8), boxprops=dict(lw=0.6), whiskerprops=dict(lw=0.6),
                    capprops=dict(lw=0.6))
    bp["boxes"][0].set_facecolor(COL[key(c)])
    vc = np.clip(v, *YL)
    ax.scatter(i + rng.uniform(-0.12, 0.12, v.size), vc, s=4, color="0.25", alpha=0.8, lw=0, zorder=3)
    r = S.loc[c]
    ax.text(i, 2.75, F.stars(r.p_signrank), ha="center",
            va="top", fontsize=5.5, linespacing=1.1, fontweight="bold" if c == "d2_bg" else "normal")
ax.axhline(0, ls="--", lw=0.5, color="k")
ax.set_ylim(YL[0] - 0.1, 2.8); ax.set_yticks([-2, -1, 0, 1, 2]); ax.set_xlim(-0.6, len(ORDER) - 0.4)   # room for the stars
ax.set_xticks(range(len(ORDER))); ax.set_xticklabels([LAB[c] for c in ORDER], rotation=30, ha="right")
ax.set_ylabel("Effect on log-odds\nBARRs : SWRs")      # per within-session SD, NREM: in the legend
F.settle(fig)
F.drop_counts(fig)
F.show(F.save(fig, "FigS6_A_barrage_share_by_measure"))

# ---- b, c: robustness (r16_robustness_index.py). Boxes = the session-level cluster bootstrap of each index
# (2 000 resamples of the 11 sessions, blocks travelling with them; the script's own fixed-seed functions);
# box = IQR, whiskers = 2.5-97.5th percentile; diamond = the full-data estimate.
import io
_txt = open(os.path.join(LOG, "r16_robustness_index.csv"), encoding="utf-8").read().splitlines()
_hb = next(i for i, l in enumerate(_txt) if l.startswith("model,"))     # the file holds part A, then part B
A = pd.read_csv(io.StringIO("\n".join(_txt[:_hb])))
B = pd.read_csv(io.StringIO("\n".join(_txt[_hb:])))
CC = pd.read_csv(ORTHO_CSV)
L = pd.read_csv(os.path.join(LOG, "r16_robustness_index_loso.csv"))
per = {m: g.drop(columns="model") for m, g in L.groupby("model")}
A2, VA = RI.boot_A(A.drop(columns=["mu_over_tau_lo", "mu_over_tau_hi", "boot_dropped"]), CC)
B2, VB = RI.boot_B(B.drop(columns=[c for c in ("transfer_ratio_lo", "transfer_ratio_hi", "boot_dropped") if c in B.columns]), per)
assert np.allclose(A2.mu_over_tau_lo, A.mu_over_tau_lo) and np.allclose(A2.mu_over_tau_hi, A.mu_over_tau_hi), "A CIs"
assert np.allclose(B2.transfer_ratio_lo.astype(float), B.transfer_ratio_lo.astype(float)), "B CIs"
print("b: " + "; ".join(f"{r.label} |mu|/tau {r.mu_over_tau:.2f} [{r.mu_over_tau_lo:.2f}, {r.mu_over_tau_hi:.2f}]"
                        for _, r in A.iterrows()))
print("c: " + "; ".join(f"{r.model} {float(r.transfer_ratio):.2f} [{float(r.transfer_ratio_lo):.2f}, "
                        f"{float(r.transfer_ratio_hi):.2f}]" for _, r in B.iterrows()))


# brackets d2 vs each other condition, b only (the paired session bootstrap) -- each resample
# draws one set of 11 sessions (blocks travel with them) and both indices are computed on it, since both are estimated
# on the same sessions; p = 2 x the smaller tail of the difference (d2 - other) around 0, two-sided, Holm across the
# comparisons of the panel (as Fig. 2e); floor 1/2000. A d2 |mu|/tau of inf (tau = 0 in some resamples)
# counts as larger. The unpaired comparison of the two boxes (P(d2 value <= other value) over all 2 000 x 2 000 pairs;
# more conservative because it ignores that both come from the same sessions) is kept in the
# table as p_unpaired. Not a rank or t-test on the bootstrap values: they are resamples of the same 11 sessions, not
# independent observations, and such a p falls to 0 as the number of resamples grows.
from r16_predictor_heterogeneity import meta
from statsmodels.stats.multitest import multipletests
preds = list(A.predictor)                                  # d2 first, then MUA, E/I, Fano, synchrony (panel b's order)
sess = sorted(CC[CC.predictor == "d2_bg"].session.unique())
byA = {c: {s: (g.coef.to_numpy(), g.se.to_numpy()) for s, g in CC[CC.predictor == c].groupby("session")} for c in preds}
byB = {m: {s: g[["heldout", "insample"]].to_numpy() for s, g in per[m].groupby("session")} for m in B.model}


def _mt(c, d):
    m = meta(np.concatenate([byA[c][s][0] for s in d]), np.concatenate([byA[c][s][1] for s in d]))
    return abs(m["re_coef"]) / m["tau"] if m["tau"] > 0 else np.inf


def _tr(mo, d):
    X = np.vstack([byB[mo][s] for s in d]); ins = np.median(X[:, 1])
    return np.median(X[:, 0]) / ins if ins > 0 else np.nan


rng_b = np.random.default_rng(0)
DA = {c: [] for c in preds[1:]}; DB = {m: [] for m in list(B.model)[1:]}
for _ in range(2000):
    d = rng_b.choice(sess, size=len(sess), replace=True)
    a0, b0 = _mt("d2_bg", d), _tr("d2 alone", d)
    for c in preds[1:]:
        DA[c].append(a0 - _mt(c, d))
    for m in DB:
        DB[m].append(b0 - _tr(m, d))


def _p(x):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    return max(min(1.0, 2 * min(np.mean(x <= 0), np.mean(x >= 0))), 1 / 2000)


def _p_unpaired(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    d = x[~np.isnan(x)][:, None] - y[~np.isnan(y)][None, :]
    return _p(d.ravel())


rows = []
for panel, D, V, ref in (("b", DA, VA, "d2_bg"), ("c", DB, VB, "d2 alone")):
    pu = [_p_unpaired(V[ref], V[k]) for k in D]
    pp = [_p(v) for v in D.values()]
    for k, a, ah, b, bh in zip(D, pu, multipletests(pu, method="holm")[1], pp, multipletests(pp, method="holm")[1]):
        rows.append(dict(panel=panel, comparison=f"d2 - {k}", full_data_diff=np.nan,
                         p_unpaired=a, p_unpaired_holm=float(ah), p_paired=b, p_paired_holm=float(bh)))
TESTS = pd.DataFrame(rows)
TESTS["full_data_diff"] = [A.set_index("predictor").mu_over_tau["d2_bg"] - A.set_index("predictor").mu_over_tau[k[5:]]
                           if p == "b" else float(B.set_index("model").transfer_ratio["d2 alone"])
                           - float(B.set_index("model").transfer_ratio[k[5:]]) for p, k in zip(TESTS.panel, TESTS.comparison)]
TESTS.to_csv(F.assert_local_path(os.path.join(LOG, "figs6_bracket_tests.csv")), index=False)
print("b brackets, paired session bootstrap, two-sided, Holm-corrected: "
      + "; ".join(f"{r.comparison} p = {r.p_paired_holm:.3g}" for _, r in TESTS[TESTS.panel == "b"].iterrows()))


def ptxt(p):                                               # as Fig. 2e's brackets: "p = 0.004 **"
    return ("p < 0.001" if p < 0.001 else f"p = {p:.3f}" if p < 0.1 else f"p = {p:.2f}") + f" {F.stars(p)}"


def brackets(ax, comps, y0, dy):                           # b only: p and stars
    for j, (x1, p) in enumerate(comps):
        F.bracket(ax, 0, x1, y0 + j * dy, ptxt(p), h=0.18 * dy, fs=5.5)


def boxes(ax, data, colors, est, bold, labels=None):
    bp = ax.boxplot(data, positions=range(len(data)), widths=0.5, whis=(2.5, 97.5), showfliers=False,
                    patch_artist=True, zorder=2, medianprops=dict(color="k", lw=0.8), whiskerprops=dict(lw=0.6),
                    capprops=dict(lw=0.6), boxprops=dict(lw=0.6))
    for patch, c in zip(bp["boxes"], colors):
        patch.set_facecolor(c)
    for i, (v, d) in enumerate(zip(est, data)):
        ax.plot(i, v, "D", ms=3, color="white", mec="k", mew=0.6, zorder=4)
        top = np.percentile(d, 97.5)                    # the value above the upper whisker, clear of the neighbours
        ax.text(i, top, f"{v:.2f}" + (labels[i] if labels else ""), ha="center", va="bottom", fontsize=5,
                fontweight="bold" if bold[i] else "normal")


fig, ax = plt.subplots(figsize=(66 * F.MM, 50 * F.MM), layout="constrained")
preds = list(A.predictor)
boxes(ax, [np.clip(VA[c], 0, 4.5) for c in preds], [COL[key(c)] for c in preds], A.mu_over_tau.to_numpy(),
      [c == "d2_bg" for c in preds])
ax.axhline(0, lw=0.5, color="k")
ax.set_xticks(range(len(preds)))
ax.set_xticklabels([LAB[c] for c in preds], rotation=30, ha="right")   # plain names as a; the residualisation in the legend
ax.set_xlim(-0.6, len(preds) - 0.4)
TB = TESTS[TESTS.panel == "b"]
brackets(ax, [(i + 1, p) for i, p in enumerate(TB.p_paired_holm)], 3.9, 0.42)
ax.set_ylim(-0.15, 3.9 + 4 * 0.42)
ax.set_ylabel("Stability across epochs\n(|μ| / τ)")
# no right axis: Φ(|μ|/τ), the expected fraction of epochs sharing the sign, goes in the legend instead.
# Φ is monotone in |μ|/τ, so the brackets' p-values are the same on either scale.
F.show(F.save(fig, "FigS6_B_stability_across_epochs"))

fig, ax = plt.subplots(figsize=(44 * F.MM, 50 * F.MM), layout="constrained")
CMAP = {"d2 alone": COL["d2"], "MUA alone": COL["MUA"], "d2 + MUA": "#b5b5b5", "all five": "#4d4d4d"}
ax.axhspan(1.0, 2.0, color="0.93", zorder=0)
ax.axhline(1.0, ls="--", lw=0.5, color="k", zorder=1); ax.axhline(0.0, lw=0.5, color="k", zorder=1)
boxes(ax, [VB[m] for m in B.model], [CMAP[m] for m in B.model], B.transfer_ratio.astype(float).to_numpy(),
      [m == "d2 alone" for m in B.model])
ax.text(len(B) - 0.45, 1.02, "1 = transferable", ha="right", va="bottom", fontsize=5, color="0.35")
ax.text(-0.55, 0.03, "nothing transfers", ha="left", va="bottom", fontsize=5, color="0.35")
ax.set_xticks(range(len(B))); ax.set_xticklabels(list(B.model), rotation=30, ha="right")
ax.set_xlim(-0.6, len(B) - 0.2)
# no brackets on c: c shows how much of d2's in-sample fit carries to a new session (the box and
# its 95 % whiskers); it is read descriptively rather than tested against the reference line
ax.set_ylim(-0.4, 1.2)
ax.set_yticks([-0.4, 0, 0.4, 0.8, 1.2])
ax.set_ylabel("Fraction of fit retained\nin a new session")      # held-out ÷ in-sample deviance explained: legend
F.show(F.save(fig, "FigS6_C_transfer_to_new_session"))